In [0]:
import pyspark.pandas as ps

In [0]:
path = 'dbfs:/FileStore/tables/arquivo_curso/dados/Spotify_Dataset/dados_tratados/data.parquet'
df_data = ps.read_parquet(path)

# Tratamento de dados

In [0]:
df_data.info()


<class 'pyspark.pandas.frame.DataFrame'>
Int64Index: 170653 entries, 0 to 170652
Data columns (total 19 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   valence           170653 non-null  float64
 1   year              170653 non-null  int32  
 2   acousticness      170653 non-null  float64
 3   artists           170653 non-null  object 
 4   danceability      170080 non-null  float64
 5   duration_ms       170454 non-null  int64  
 6   energy            170573 non-null  float64
 7   explicit          170606 non-null  int64  
 8   id                170653 non-null  object 
 9   instrumentalness  170270 non-null  float64
 10  key               170451 non-null  int64  
 11  liveness          170613 non-null  float64
 12  loudness          170621 non-null  float64
 13  mode              170635 non-null  int64  
 14  name              170653 non-null  object 
 15  popularity        169496 non-null  int64  
 16  release_date     

Revomendo os dados Nulos

In [0]:
df_data = df_data.dropna()

In [0]:
df_data.info()

<class 'pyspark.pandas.frame.DataFrame'>
Int64Index: 169300 entries, 0 to 170652
Data columns (total 19 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   valence           169300 non-null  float64
 1   year              169300 non-null  int32  
 2   acousticness      169300 non-null  float64
 3   artists           169300 non-null  object 
 4   danceability      169300 non-null  float64
 5   duration_ms       169300 non-null  int64  
 6   energy            169300 non-null  float64
 7   explicit          169300 non-null  int64  
 8   id                169300 non-null  object 
 9   instrumentalness  169300 non-null  float64
 10  key               169300 non-null  int64  
 11  liveness          169300 non-null  float64
 12  loudness          169300 non-null  float64
 13  mode              169300 non-null  int64  
 14  name              169300 non-null  object 
 15  popularity        169300 non-null  int64  
 16  release_date     

In [0]:
df_data['artist_song'] = df_data.artists + ' - ' + df_data.name

In [0]:
df_data.head()


,valence,year,acousticness,artists,danceability,duration_ms,energy,explicit,id,instrumentalness,key,liveness,loudness,mode,name,popularity,release_date,speechiness,tempo,artist_song
0,0.917,1970,0.096000,The Velvet Underground,0.624,201440,0.774,0,60ZyiL4lmWzZyGfqyECTqp,0.030900,7,0.0960,-10.391,1,Train Round the Bend - 2015 Remaster,24,1970,0.0315,117.006,The Velvet Underground - Train Round the Bend ...
1,0.511,1970,0.001900,Ten Years After,0.405,458463,0.543,0,6DYyyUdHzI6RdSx0swUR1i,0.720000,2,0.1860,-9.313,1,Love Like a Man - 2017 Remaster,34,1970-04-01,0.0290,107.598,Ten Years After - Love Like a Man - 2017 Remaster
2,0.466,1970,0.052800,The Mothers Of Invention,0.444,105587,0.568,0,6HJAS8XZO0ctUcN2KsbLRa,0.000010,11,0.5120,-8.800,0,Oh No,24,1970-08-10,0.0327,124.319,The Mothers Of Invention - Oh No
3,0.523,1970,0.081100,Three Dog Night,0.502,174707,0.669,0,7sZ74qmKb1nyGKUgHROJ1n,0.000945,7,0.0906,-11.725,1,One Man Band,19,1970-01-01,0.0912,121.089,Three Dog Night - One Man Band
4,0.501,1970,0.000128,The Rolling Stones,0.273,246413,0.866,0,095WtNlSHE8TMB2gQ1fdTx,0.790000,11,0.9610,-7.598,1,Street Fighting Man - Live,25,1970-09-04,0.0347,134.891,The Rolling Stones - Street Fighting Man - Live


In [0]:
df_data.info()

<class 'pyspark.pandas.frame.DataFrame'>
Int64Index: 169300 entries, 0 to 170652
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   valence           169300 non-null  float64
 1   year              169300 non-null  int32  
 2   acousticness      169300 non-null  float64
 3   artists           169300 non-null  object 
 4   danceability      169300 non-null  float64
 5   duration_ms       169300 non-null  int64  
 6   energy            169300 non-null  float64
 7   explicit          169300 non-null  int64  
 8   id                169300 non-null  object 
 9   instrumentalness  169300 non-null  float64
 10  key               169300 non-null  int64  
 11  liveness          169300 non-null  float64
 12  loudness          169300 non-null  float64
 13  mode              169300 non-null  int64  
 14  name              169300 non-null  object 
 15  popularity        169300 non-null  int64  
 16  release_date     

Separando todas as variavies disponivel para o ML

In [0]:
X = df_data.columns.to_list()
X.remove('artists')
X.remove('id')
X.remove('name')
X.remove('artist_song')
X.remove('release_date')
X



Out[16]: ['valence',
 'year',
 'acousticness',
 'danceability',
 'duration_ms',
 'energy',
 'explicit',
 'instrumentalness',
 'key',
 'liveness',
 'loudness',
 'mode',
 'popularity',
 'speechiness',
 'tempo']

Convertendo para Spark para conseguir usar a ML Lib

In [0]:
df_data = df_data.to_spark()

# Vetorização

In [0]:
from pyspark.ml.feature import VectorAssembler

In [0]:
dados_encoder_vector = VectorAssembler (inputCols = X, outputCol='features').transform(df_data)

In [0]:
dados_encoder_vector.select('features').show(truncate=False, n=5)

+----------------------------------------------------------------------------------------------------------------------+
|features                                                                                                              |
+----------------------------------------------------------------------------------------------------------------------+
|[0.917,1970.0,0.096,0.624,201440.0,0.774,0.0,0.0309,7.0,0.096,-10.390999999999998,1.0,24.0,0.0315,117.006]            |
|[0.511,1970.0,0.0019,0.405,458463.0,0.5429999999999999,0.0,0.72,2.0,0.18600000000000005,-9.313,1.0,34.0,0.029,107.598]|
|[0.466,1970.0,0.0528,0.444,105587.0,0.568,0.0,1.02E-5,11.0,0.512,-8.8,0.0,24.0,0.0327,124.319]                        |
|[0.523,1970.0,0.0811,0.502,174707.0,0.669,0.0,9.45E-4,7.0,0.0906,-11.725,1.0,19.0,0.0912,121.089]                     |
|[0.501,1970.0,1.28E-4,0.273,246413.0,0.866,0.0,0.79,11.0,0.961,-7.598,1.0,25.0,0.0347,134.891]                        |
+-------------------------------

# Padronização

In [0]:
from pyspark.ml.feature import StandardScaler

In [0]:

# Supondo que você já tenha um DataFrame chamado 'dados_encoded_vector' com a coluna 'features'
scaler = StandardScaler(inputCol='features', outputCol='features_scaled')

# Ajustando o scaler aos dados
model_scaler = scaler.fit(dados_encoder_vector)

# Transformando os dados
dados_musicas_scaler = model_scaler.transform(dados_encoder_vector)

# Visualizando os dados escalados
dados_musicas_scaler.select('features_scaled').show(truncate=False, n=5)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|features_scaled                                                                                                                                                                                                                                                               |
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|[3.493845627581973,76.01945247914719,0.25560097373899326,3.553867728626555,1.6031732036099804,2.8949505209817366,0.0,0.09859458986370954,1.9913197029800866,0.5493560155028588,-1.83